# 04 | Reconcile and validate
A failing assertion stops the notebook so a Fabric pipeline can report failure. Expected counts vary by which batches were ingested; reconciliation works for either state.

In [ ]:
from pyspark.sql import functions as F
bronze = spark.table("bronze_orders")
silver = spark.table("silver_orders")
quarantine = spark.table("silver_orders_quarantine")
gold = spark.table("gold_daily_sales")
latest_count = bronze.select("order_id").distinct().count()
assert bronze.count() in (1604,2006), "Unexpected Bronze row count for the provided demo files"
assert silver.count() + quarantine.count() == latest_count, "Silver+quarantine must reconcile to distinct Bronze orders"
assert silver.filter((F.col("quantity") <= 0) | (F.col("unit_price") <= 0)).count() == 0
assert silver.groupBy("order_id").count().filter("count > 1").count() == 0
completed = silver.filter("status = 'Completed'")
sales_total = completed.agg(F.sum("line_amount")).first()[0]
gold_total = gold.agg(F.sum("revenue")).first()[0]
assert abs(float(sales_total or 0) - float(gold_total or 0)) < .01, "Daily sales revenue does not reconcile"
assert quarantine.count() >= 4, "Expected rejected demo records missing"
print({"bronze_rows":bronze.count(),"distinct_order_ids":latest_count,
       "silver_rows":silver.count(),"quarantine_rows":quarantine.count(),
       "gold_revenue":str(gold_total),"checks":"PASS"})
